# 02 - Data Cleaning

Applies the documented cleaning pipeline from `src/data_cleaning.py` to the raw dataset and verifies the result.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import config
from data_loader import load_csv, save_csv
from data_cleaning import (standardize_gender, drop_duplicates, impute_total_charges,
                            enforce_dtypes, flag_outliers_iqr, clean_pipeline)

df = load_csv(config.RAW_DATA_PATH)
print('Before cleaning:', df.shape)

2026-07-07 08:12:39,037 | INFO | Loaded customer_churn.csv: 7525 rows x 27 columns


Before cleaning: (7525, 27)


## Step 1: Standardize Gender casing

In [2]:
df = standardize_gender(df)
df['Gender'].unique()

<StringArray>
['Female', 'Male']
Length: 2, dtype: str

## Step 2: Remove duplicate customer records

In [3]:
before = len(df)
df = drop_duplicates(df)
print(f'Removed {before - len(df)} duplicate rows')

2026-07-07 08:12:39,082 | INFO | Removed 25 duplicate rows


Removed 25 duplicate rows


## Step 3: Impute missing TotalCharges

Rather than a generic median fill, we use `Tenure x MonthlyCharges` — a domain-accurate estimate of what the customer would have been billed.

In [4]:
missing_before = df['TotalCharges'].isna().sum()
df = impute_total_charges(df)
print(f'Imputed {missing_before} values. Remaining missing:', df['TotalCharges'].isna().sum())

2026-07-07 08:12:39,091 | INFO | Imputed 37 missing TotalCharges values using Tenure x MonthlyCharges


Imputed 37 values. Remaining missing: 0


## Step 4: Enforce clean data types

In [5]:
df = enforce_dtypes(df)
df.dtypes

CustomerID                              str
Gender                                  str
Age                                   int64
SeniorCitizen                         int64
MaritalStatus                           str
Dependents                              str
Tenure                                int64
ContractType                            str
InternetService                         str
PhoneService                            str
StreamingTV                             str
StreamingMovies                         str
OnlineSecurity                          str
OnlineBackup                            str
DeviceProtection                        str
TechSupport                             str
MonthlyCharges                      float64
TotalCharges                        float64
PaymentMethod                           str
PaperlessBilling                        str
NumberOfComplaints                    int64
CustomerSatisfactionScore             int64
SupportTickets                  

## Step 5: Flag outliers (IQR method) — flag, don't blindly drop

Business-plausible extreme values (e.g. high-tier bundle customers) are flagged in a new column rather than removed, so the modeling step can decide how to treat them.

In [6]:
df = flag_outliers_iqr(df)
df[['MonthlyCharges_Outlier','Tenure_Outlier','CustomerLifetimeValue_Outlier']].sum()

2026-07-07 08:12:39,115 | INFO | MonthlyCharges: flagged 0 outliers (IQR bounds [14.41, 149.16])


2026-07-07 08:12:39,119 | INFO | Tenure: flagged 0 outliers (IQR bounds [-36.00, 92.00])


2026-07-07 08:12:39,122 | INFO | CustomerLifetimeValue: flagged 75 outliers (IQR bounds [-3207.83, 10272.66])


MonthlyCharges_Outlier            0
Tenure_Outlier                    0
CustomerLifetimeValue_Outlier    75
dtype: int64

## Final validation checks

In [7]:
assert df['Churn'].isin(['Yes','No']).all()
assert df['TotalCharges'].isna().sum() == 0
assert df['CustomerID'].is_unique
print('All validation checks passed.')
print('Final shape:', df.shape)

All validation checks passed.
Final shape: (7500, 30)


In [8]:
save_csv(df, config.CLEAN_DATA_PATH)
df.head()

2026-07-07 08:12:39,224 | INFO | Saved customer_churn_clean.csv: 7500 rows x 30 columns


,CustomerID,Gender,Age,SeniorCitizen,MaritalStatus,Dependents,Tenure,ContractType,InternetService,PhoneService,...,NumberOfComplaints,CustomerSatisfactionScore,SupportTickets,LastInteractionDate,ReferralStatus,CustomerLifetimeValue,Churn,MonthlyCharges_Outlier,Tenure_Outlier,CustomerLifetimeValue_Outlier
0,CUST-100000,Female,40,0,Married,Yes,22,Month-to-Month,Fiber Optic,No,...,1,3,2,2025-10-16,No,2724.64,Yes,0,0,0
1,CUST-100001,Male,23,0,Married,Yes,43,Month-to-Month,No,No,...,1,4,2,2025-06-29,Yes,1213.14,No,0,0,0
2,CUST-100002,Female,38,0,Widowed,No,28,Month-to-Month,DSL,Yes,...,0,3,2,2026-01-22,No,3242.81,No,0,0,0
3,CUST-100003,Female,27,0,Married,No,1,Month-to-Month,Fiber Optic,Yes,...,1,3,4,2026-04-14,Yes,1697.67,Yes,0,0,0
4,CUST-100004,Male,37,0,Single,Yes,0,Month-to-Month,No,Yes,...,2,3,5,2025-09-27,No,430.29,No,0,0,0
